In [1]:
import os
import jax
import flax
import tyro
import time
import optax
import wandb
import pickle
import random
import wandb_osh
import numpy as np
import flax.linen as nn
import jax.numpy as jnp

from brax import envs
from etils import epath
from dataclasses import dataclass
from collections import namedtuple
from typing import NamedTuple, Any
from wandb_osh.hooks import TriggerWandbSyncHook
from flax.training.train_state import TrainState
from flax.linen.initializers import variance_scaling
from brax.io import html

from evaluator import CrlEvaluator
from buffer import TrajectoryUniformSamplingQueue
from memory_bank import MemoryBank, MemoryBankState

from pathlib import Path
import glob

In [2]:
width = 256
depth = 1024

In [ ]:
lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
bias_init = nn.initializers.zeros
def residual_block(x, width, normalize, activation):
    identity = x
    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)
    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)
    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)
    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)
    x = x + identity
    return x
class SA_encoder(nn.Module):
    norm_type = "layer_norm"
    network_width: int = 1024
    network_depth: int = 4
    skip_connections: int = 0
    use_relu: int = 0
    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray):

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros
        
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x
        
        if self.use_relu:
            activation = nn.relu
        else:
            activation = nn.swish
            
        x = jnp.concatenate([s, a], axis=-1)
        #Initial layer
        x = nn.Dense(self.network_width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)
        #Residual blocks
        for i in range(self.network_depth // 4):
            x = residual_block(x, self.network_width, normalize, activation)
        #Final layer
        x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        return x

In [4]:
seed = 999

In [5]:
key = jax.random.PRNGKey(seed)
key, buffer_key, env_key, eval_env_key, actor_key, sa_key, g_key, sym_key, asym_key, memory_bank_key = jax.random.split(key, 10)

In [5]:
# sa_encoder = SA_encoder(network_width=width, network_depth=depth, skip_connections=4, use_relu=0)

In [6]:
# sa_encoder_params = sa_encoder.init(sa_key, np.ones([1, 268]), np.ones([1, 17]))

In [6]:
#Run this:

# Instantiate the network:
net = SA_encoder(
    network_width=width, 
    network_depth=depth, 
    skip_connections=4, 
    use_relu=False
)

# Create random seeds
key = jax.random.PRNGKey(0)
key, sa_key = jax.random.split(key)

# Initialize parameters
# (shape of s is [batch_size, 268], shape of a is [batch_size, 17])
params = net.init(sa_key, jnp.ones([1, 268]), jnp.ones([1, 17]))

# # Forward pass (un-jitted)
# output = net.apply(params, jnp.ones([1, 268]), jnp.ones([1, 17]))
# print("Output (un-jitted) shape:", output.shape)

# Optionally jit-compile for faster execution
compiled_forward = jax.jit(net.apply)
output_jitted = compiled_forward(params, jnp.ones([1, 268]), jnp.ones([1, 17]))
print("Output (jitted) shape:", output_jitted.shape)


TypeError: scan() got an unexpected keyword argument 'f'

In [3]:
class ResidualBlock(nn.Module):
    width: int
    norm_type: str = "layer_norm"
    use_relu: bool = False  # allows toggling between nn.relu and nn.swish

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        # Choose normalization
        if self.norm_type == "layer_norm":
            normalize = nn.LayerNorm()
        else:
            normalize = lambda z: z  # no-op if unspecified

        # Choose activation
        activation = nn.relu if self.use_relu else nn.swish

        # We'll use the same kernel initialization and bias init
        # that were used originally.
        lecun_uniform = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        # First half of the residual path
        identity = x
        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)

        # Second half of the residual path
        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)

        # Combine with identity
        return x + identity

class SA_encoder(nn.Module):
    norm_type: str = "layer_norm"
    network_width: int = 1024
    network_depth: int = 4       # number of times we'll stack the block
    skip_connections: int = 0    # leaving in case you want to expand skip logic
    use_relu: int = 0            # toggle relu vs swish

    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray) -> jnp.ndarray:
        # Same kernel init / bias init as before
        lecun_uniform = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        # Choose normalization
        if self.norm_type == "layer_norm":
            normalize = nn.LayerNorm()
        else:
            normalize = lambda x: x  # use a no-op if needed

        # Choose activation
        activation = nn.relu if self.use_relu else nn.swish

        # Concatenate input
        x = jnp.concatenate([s, a], axis=-1)

        # Initial dense layer
        x = nn.Dense(self.network_width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)

        # Residual blocks (using our new ResidualBlock module).
        # Note: If you want more layering, you can increase network_depth
        for _ in range(self.network_depth):
            x = ResidualBlock(
                width=self.network_width, 
                norm_type=self.norm_type,
                use_relu=self.use_relu
            )(x)

        # Final output layer
        x = nn.Dense(64, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        return x

In [3]:
import jax
import jax.numpy as jnp
from jax import lax
import flax.linen as nn
from flax.linen.initializers import variance_scaling

class ResidualBlock(nn.Module):
    width: int
    norm_type: str = "layer_norm"
    use_relu: bool = False  # Allows toggling between nn.relu and nn.swish

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        # Choose normalization
        if self.norm_type == "layer_norm":
            normalize = nn.LayerNorm()
        else:
            normalize = lambda z: z  # no-op

        # Choose activation
        activation = nn.relu if self.use_relu else nn.swish

        # Setup initialization
        lecun_uniform = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        identity = x
        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)

        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)

        # Combine with identity
        return x + identity


class ResidualBlockStack(nn.Module):
    """
    Scans over a single ResidualBlock multiple times.
    """
    width: int
    norm_type: str
    use_relu: bool
    network_depth: int

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        """
        Applies ResidualBlock repeatedly using jax.lax.scan.
        """

        def scan_fn(carry, _):
            # carry is the running 'x' 
            #  _ is a dummy (we don't need an explicit input for each iteration)
            new_carry = ResidualBlock(
                width=self.width,
                norm_type=self.norm_type,
                use_relu=self.use_relu
            )(carry)
            return new_carry, None

        # repeated application of ResidualBlock network_depth times
        x, _ = lax.scan(scan_fn, x, None, length=self.network_depth)
        return x


class SA_encoder(nn.Module):
    norm_type: str = "layer_norm"
    network_width: int = 1024
    network_depth: int = 4
    skip_connections: int = 0  # left in place if you need it later
    use_relu: int = 0

    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray) -> jnp.ndarray:
        lecun_uniform = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        # Choose normalization
        if self.norm_type == "layer_norm":
            normalize = nn.LayerNorm()
        else:
            normalize = lambda x: x

        # Choose activation
        activation = nn.relu if self.use_relu else nn.swish

        # Concatenate input
        x = jnp.concatenate([s, a], axis=-1)

        # Initial dense layer
        x = nn.Dense(self.network_width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)

        # Use our ResidualBlockStack to scan over blocks
        x = ResidualBlockStack(
            width=self.network_width,
            norm_type=self.norm_type,
            use_relu=bool(self.use_relu),
            network_depth=self.network_depth
        )(x)

        # Final output layer
        x = nn.Dense(64, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        return x

In [5]:
import jax
import jax.numpy as jnp

def run_forward_pass():
    # 1) Create a PRNGKey for initialization
    seed = 999
    key = jax.random.PRNGKey(seed)

    # 2) Split off the key for network parameter initialization
    key, sa_key = jax.random.split(key, 2)

    # 3) Instantiate your SA_encoder with desired hyperparameters
    width = 256
    depth = 4
    sa_encoder = SA_encoder(
        network_width=width,
        network_depth=depth,
        skip_connections=4,  # if you're using skip connections
        use_relu=0           # change to 1 if you like ReLU
    )

    # 4) Initialize model parameters with dummy input shapes
    #    Make sure the dummy shapes match your real input (here, [1,268], [1,17])
    sa_encoder_params = sa_encoder.init(sa_key, jnp.ones([1, 268]), jnp.ones([1, 17]))

    # 5) Optionally JIT-compile the apply function
    sa_encoder_apply = jax.jit(sa_encoder.apply)

    # 6) Run a forward pass with the same dummy inputs
    output = sa_encoder_apply(sa_encoder_params, jnp.ones([1, 268]), jnp.ones([1, 17]))
    print("Output shape:", output.shape)
    print("Output values:\n", output)

# Example usage (in a notebook or script):
run_forward_pass()

JaxTransformError: Jax transforms and Flax models cannot be mixed. (https://flax.readthedocs.io/en/latest/api_reference/flax.errors.html#flax.errors.JaxTransformError)

In [5]:
lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
bias_init = nn.initializers.zeros
def residual_block(x, width, normalize, activation):
    identity = x
    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)

    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)

    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)

    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)

    x = x + identity
    return x

class SA_encoder(nn.Module):
    norm_type: str = "layer_norm"
    network_width: int = 1024
    network_depth: int = 4
    skip_connections: int = 0
    use_relu: int = 0

    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray):
        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros
        
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x
        
        if self.use_relu:
            activation = nn.relu
        else:
            activation = nn.swish
            
        x = jnp.concatenate([s, a], axis=-1)
        # Initial layer
        x = nn.Dense(self.network_width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)

        # Prepare the scanning body for repeated residual blocks
        def block_step(carry, _):
            out = residual_block(
                carry, 
                self.network_width, 
                normalize, 
                activation
            )
            return out, None

        # Use jax.lax.scan to apply the same block multiple times
        x, _ = jax.lax.scan(
            block_step, 
            x, 
            None, 
            length=self.network_depth // 4
        )

        # Final layer
        x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        return x

In [3]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.linen.initializers import variance_scaling

lecun_uniform = variance_scaling(1/3, "fan_in", "uniform")
bias_init = nn.initializers.zeros

class ResidualBlock(nn.Module):
    """A single 4×Dense residual block with normalization + activation."""
    width: int
    activation: callable
    norm_factory: callable  # e.g., lambda: nn.LayerNorm()

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        identity = x
        
        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = self.norm_factory()(x)
        x = self.activation(x)

        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = self.norm_factory()(x)
        x = self.activation(x)

        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = self.norm_factory()(x)
        x = self.activation(x)

        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = self.norm_factory()(x)
        x = self.activation(x)

        return x + identity

class SA_encoder(nn.Module):
    """An example Flax module applying multiple ResidualBlocks via jax.lax.scan."""
    norm_type: str = "layer_norm"
    network_width: int = 1024
    network_depth: int = 4
    skip_connections: int = 0  # not used in this snippet
    use_relu: int = 0

    def setup(self):
        # Choose normalization
        if self.norm_type == "layer_norm":
            # We'll create new instances of LayerNorm inside each block
            # so we store a factory here, e.g. lambda: nn.LayerNorm
            self.norm_factory = lambda: nn.LayerNorm
        else:
            # if no normalization, just identity
            # we still must provide a callable returning a no-op
            self.norm_factory = lambda: lambda: lambda x: x
        
        # Activation selection
        self.activation = nn.relu if self.use_relu else nn.swish
        
        # The initial Dense layer
        self.initial_dense = nn.Dense(self.network_width, 
                                      kernel_init=lecun_uniform, 
                                      bias_init=bias_init)
        
        # Create as many ResidualBlock submodules as needed
        # Each block is a full stacked 4×Dense residual pattern
        n_blocks = self.network_depth // 4
        self.blocks = [
            ResidualBlock(width=self.network_width, 
                          activation=self.activation, 
                          norm_factory=self.norm_factory)
            for _ in range(n_blocks)
        ]
        
        # The final Dense layer
        self.final_dense = nn.Dense(64, kernel_init=lecun_uniform, bias_init=bias_init)

    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray) -> jnp.ndarray:
        # Concatenate s and a
        x = jnp.concatenate([s, a], axis=-1)
        
        # Initial layer + normalization + activation
        x = self.initial_dense(x)
        x = self.norm_factory()(x)  # create a LayerNorm or identity
        x = self.activation(x)

        # We define a scanning function that just calls each residual block
        def scan_block(carry, block):
            new_carry = block(carry)
            return new_carry, None

        # Use jax.lax.scan to apply all submodules in self.blocks
        x, _ = jax.lax.scan(scan_block, x, self.blocks)

        # Final layer
        x = self.final_dense(x)
        return x

In [5]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.linen.initializers import variance_scaling

lecun_uniform = variance_scaling(1/3, "fan_in", "uniform")
bias_init = nn.initializers.zeros

########################################
# 1) Define a ResidualBlock as a submodule
########################################
class ResidualBlock(nn.Module):
    width: int
    activation: callable
    use_layernorm: bool

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        # Each "residual block" contains multiple Dense + Norm + Activation steps
        # If we want layernorm, define it as a submodule here:
        def maybe_norm(y):
            return nn.LayerNorm()(y) if self.use_layernorm else y

        identity = x

        # Dense 1
        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = maybe_norm(x)
        x = self.activation(x)

        # Dense 2
        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = maybe_norm(x)
        x = self.activation(x)

        return x + identity

########################################
# 2) Define the main encoder
########################################
class SAEncoder(nn.Module):
    network_width: int = 1024
    network_depth: int = 4
    use_layernorm: bool = True
    use_relu: bool = False

    def setup(self):
        # Choose activation
        self.activation = nn.relu if self.use_relu else nn.swish

        # Initial Dense
        self.initial_dense = nn.Dense(
            self.network_width, 
            kernel_init=lecun_uniform, 
            bias_init=bias_init
        )

        # Create as many residual blocks as we want
        # Suppose each "depth unit" is one block:
        self.blocks = [
            ResidualBlock(
                width=self.network_width,
                activation=self.activation,
                use_layernorm=self.use_layernorm
            )
            for _ in range(self.network_depth)
        ]

        # Final Dense
        self.final_dense = nn.Dense(64, kernel_init=lecun_uniform, bias_init=bias_init)

    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray) -> jnp.ndarray:
        # Concatenate s and a
        x = jnp.concatenate([s, a], axis=-1)

        # Pass through initial Dense
        x = self.initial_dense(x)

        # (Optional) LayerNorm + activation after initial Dense
        if self.use_layernorm:
            x = nn.LayerNorm()(x)  # returns an array
        x = self.activation(x)

        # Pass through each residual block
        # (Here we use a standard Python loop to call submodules;
        #  if you prefer jax.lax.scan, see below.)
        for block in self.blocks:
            x = block(x)

        # Final layer
        x = self.final_dense(x)
        return x

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.linen.initializers import variance_scaling

##########################################################
# 1) Define Submodules
##########################################################
lecun_uniform = variance_scaling(1/3, "fan_in", "uniform")
bias_init = nn.initializers.zeros

class ResidualBlock(nn.Module):
    width: int
    activation: callable
    use_layernorm: bool

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        def maybe_norm(y):
            return nn.LayerNorm()(y) if self.use_layernorm else y

        identity = x
        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = maybe_norm(x)
        x = self.activation(x)

        x = nn.Dense(self.width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = maybe_norm(x)
        x = self.activation(x)

        return x + identity

class SAEncoder(nn.Module):
    network_width: int = 1024
    network_depth: int = 2
    use_layernorm: bool = True
    use_relu: bool = False

    def setup(self):
        self.activation = nn.relu if self.use_relu else nn.swish
        self.initial_dense = nn.Dense(
            self.network_width, 
            kernel_init=lecun_uniform, 
            bias_init=bias_init
        )
        # Create multiple residual blocks
        self.blocks = [
            ResidualBlock(
                width=self.network_width, 
                activation=self.activation, 
                use_layernorm=self.use_layernorm
            )
            for _ in range(self.network_depth)
        ]
        self.final_dense = nn.Dense(64, kernel_init=lecun_uniform, bias_init=bias_init)

    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray) -> jnp.ndarray:
        x = jnp.concatenate([s, a], axis=-1)
        x = self.initial_dense(x)
        if self.use_layernorm:
            x = nn.LayerNorm()(x)
        x = self.activation(x)

        for block in self.blocks:
            x = block(x)

        x = self.final_dense(x)
        return x

##########################################################
# 2) Create Model, Initialize, Compile, and Run Forward Pass
##########################################################
def main():
    # Instantiate the model
    net = SAEncoder(network_width=256, network_depth=1024, use_layernorm=True, use_relu=False)
    
    # Create random keys
    rng = jax.random.PRNGKey(0)
    rng_init, rng_apply = jax.random.split(rng, 2)

    # Dummy inputs
    s = jnp.ones((1, 268))
    a = jnp.ones((1, 17))

    # Initialize parameters
    params = net.init(rng_init, s, a)
    print("Initialized parameter shapes:", jax.tree_map(jnp.shape, params))

    # Forward pass (un-jitted)
    output = net.apply(params, s, a)
    print("Output (un-jitted) shape:", output.shape)

    # JIT-compile the forward pass
    compiled_forward = jax.jit(net.apply)
    output_jit = compiled_forward(params, s, a)
    print("Output (jitted) shape:", output_jit.shape)


if __name__ == "__main__":
    main()

Initialized parameter shapes: {'params': {'LayerNorm_0': {'bias': (256,), 'scale': (256,)}, 'blocks_0': {'Dense_0': {'bias': (256,), 'kernel': (256, 256)}, 'Dense_1': {'bias': (256,), 'kernel': (256, 256)}, 'LayerNorm_0': {'bias': (256,), 'scale': (256,)}, 'LayerNorm_1': {'bias': (256,), 'scale': (256,)}}, 'blocks_1': {'Dense_0': {'bias': (256,), 'kernel': (256, 256)}, 'Dense_1': {'bias': (256,), 'kernel': (256, 256)}, 'LayerNorm_0': {'bias': (256,), 'scale': (256,)}, 'LayerNorm_1': {'bias': (256,), 'scale': (256,)}}, 'blocks_10': {'Dense_0': {'bias': (256,), 'kernel': (256, 256)}, 'Dense_1': {'bias': (256,), 'kernel': (256, 256)}, 'LayerNorm_0': {'bias': (256,), 'scale': (256,)}, 'LayerNorm_1': {'bias': (256,), 'scale': (256,)}}, 'blocks_100': {'Dense_0': {'bias': (256,), 'kernel': (256, 256)}, 'Dense_1': {'bias': (256,), 'kernel': (256, 256)}, 'LayerNorm_0': {'bias': (256,), 'scale': (256,)}, 'LayerNorm_1': {'bias': (256,), 'scale': (256,)}}, 'blocks_1000': {'Dense_0': {'bias': (256,

In [ ]:
print('hi')

In [5]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.linen.initializers import variance_scaling

def residual_block_scan(
    x: jnp.ndarray,
    width: int,
    normalize,
    activation,
    lecun_uniform,
    bias_init
):
    """A single residual block step, to be used inside a scan."""
    identity = x
    # You can split these into fewer Dense calls if desired, but
    # here we match the original code's 4 operations.
    for _ in range(4):
        x = nn.Dense(width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)
    return x + identity

class SA_encoder(nn.Module):
    norm_type: str = "layer_norm"
    network_width: int = 1024
    network_depth: int = 4
    skip_connections: int = 0
    use_relu: int = 0

    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray):
        # Same initial setup
        lecun_uniform = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        if self.use_relu:
            activation = nn.relu
        else:
            activation = nn.swish

        # Concatenate s and a for the input
        x = jnp.concatenate([s, a], axis=-1)

        # Initial layer
        x = nn.Dense(self.network_width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)

        # Now we use scan to repeatedly apply our residual block
        def scan_fn(carry, _):
            new_carry = residual_block_scan(
                carry,
                self.network_width,
                normalize,
                activation,
                lecun_uniform,
                bias_init
            )
            return new_carry, None

        # Unroll network_depth//4 residual blocks
        x, _ = nn.scan(
            f=scan_fn,
            variable_broadcast="params",   # share params across steps if desired
            split_rngs={"params": False},  # do not split rng for param init each step
            in_axes=None,
            length=self.network_depth // 4
        )(x, None)

        # Final layer
        x = nn.Dense(64, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        return x

In [3]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.linen.initializers import variance_scaling

def residual_block_scan(
    x: jnp.ndarray,
    width: int,
    normalize,
    activation,
    lecun_uniform,
    bias_init
):
    """A single residual block step, to be used inside a scan."""
    identity = x
    # The original code had four Dense calls in a row:
    for _ in range(4):
        x = nn.Dense(width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)
    return x + identity

class SA_encoder(nn.Module):
    norm_type: str = "layer_norm"
    network_width: int = 1024
    network_depth: int = 4
    skip_connections: int = 0
    use_relu: int = 0

    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray):
        # Setup initializers
        lecun_uniform = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        # Choose normalization
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        # Choose activation
        if self.use_relu:
            activation = nn.relu
        else:
            activation = nn.swish

        # Concatenate s and a for the input
        x = jnp.concatenate([s, a], axis=-1)

        # Initial layer
        x = nn.Dense(self.network_width, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)

        # Use scan to repeatedly apply the residual block
        def scan_fn(carry, _):
            new_carry = residual_block_scan(
                carry,
                self.network_width,
                normalize,
                activation,
                lecun_uniform,
                bias_init
            )
            return new_carry, None

        # Pass scan_fn as the first argument, removing the f= keyword
        x, _ = nn.scan(
            scan_fn,
            variable_broadcast="params",
            split_rngs={"params": False},
            in_axes=None,
            length=self.network_depth // 4
        )(x, None)

        # Final layer
        x = nn.Dense(64, kernel_init=lecun_uniform, bias_init=bias_init)(x)
        return x

# -------------------------------
# Example usage: Construct and run
# -------------------------------
if __name__ == "__main__":
    key = jax.random.PRNGKey(42)
    rng_params, rng_input_s, rng_input_a = jax.random.split(key, 3)

    # Create random input
    s_example = jax.random.normal(rng_input_s, shape=(1, 268))
    a_example = jax.random.normal(rng_input_a, shape=(1, 17))

    # Construct the module
    sa_encoder = SA_encoder(network_width=256, network_depth=4, skip_connections=4, use_relu=0)

    # Initialize parameters
    params = sa_encoder.init(rng_params, s_example, a_example)

    # Create a jitted forward pass
    @jax.jit
    def forward_pass(params, s, a):
        return sa_encoder.apply(params, s, a)

    # Run forward pass
    output = forward_pass(params, s_example, a_example)
    print("Output shape:", output.shape)
    print("Output values:", output)

AttributeError: 'ArrayImpl' object has no attribute '_state'

In [6]:
   import flax
   print(flax.__version__)
   print(flax)

0.7.5
<module 'flax' from '/home/kw6487/.conda/envs/expl-env-flax075-jupyter/lib/python3.10/site-packages/flax/__init__.py'>


In [2]:
print('hi')

hi


In [4]:
import jax
import jax.numpy as jnp
import flax.linen as nn

def scan_fn(carry, x):
    # A trivial function that accumulates a running sum in carry
    new_carry = carry + x
    return new_carry, None

class ScanTester(nn.Module):
    @nn.compact
    def __call__(self, x):
        # Initialize carry as a JAX array (not a Python float)
        carry_init = jnp.array(0.0)

        # We scan over axis=1 of x. 
        # If x.shape is (batch, seq_length), we scan across "seq_length".
        carry_final, _ = nn.scan(
            scan_fn,
            in_axes=1,
            length=x.shape[1],
        )(carry_init, x)

        return carry_final

def main():
    key = jax.random.PRNGKey(42)
    x = jax.random.normal(key, shape=(1, 10))  # shape: (batch=1, seq_length=10)

    model = ScanTester()
    params = model.init(key, x)  # init with the same RNG
    out = model.apply(params, x)

    print("Scan output shape:", out.shape)
    print("Scan output:", out)

if __name__ == "__main__":
    main()

AttributeError: 'EvalTrace' object has no attribute 'level'

In [3]:

import jax
import jax.numpy as jnp
import flax.linen as nn

class StatefulModule(nn.Module):
    @nn.compact
    def __call__(self, x, carry):
        counter = self.variable('batch_stats', 'counter', lambda: jnp.array(0))
        new_counter = counter.value + 1
        counter.value = new_counter
        return new_counter, x + carry

inputs = jnp.arange(5)
initial_carry = 0

stateful_module = StatefulModule()
variables = stateful_module.init(jax.random.key(0), inputs[0], initial_carry)

scan_fn = lambda carry, x: stateful_module.apply(
    variables, x, carry, mutable=['batch_stats']
)

final_carry, outputs = jax.lax.scan(scan_fn, initial_carry, inputs)

print(outputs)

AttributeError: 'EvalTrace' object has no attribute 'level'

In [3]:
def test_flax_scan():
    import jax
    import jax.numpy as jnp
    from flax import linen as nn

    class RecurrentSum(nn.Module):
        @nn.compact
        def __call__(self, xs):
            # Define a simple step function: add x to carry
            def step(carry, x):
                carry = carry + x
                return carry, carry
            
            # Use nn.scan over the first axis of xs
            final_carry, ys = nn.scan(step, in_axes=0)(0.0, xs)
            return final_carry, ys

    # Create model and test input
    model = RecurrentSum()
    inputs = jnp.array([1.0, 2.0, 3.0])
    
    final_carry, outputs = model.apply({}, inputs)
    
    # Print results
    print("Final carry:", final_carry)   # Expected: 6.0
    print("Outputs:", outputs)          # Expected: [1.0, 3.0, 6.0]

    # Verify correctness
    assert jnp.allclose(final_carry, 6.0), "Final carry should be 6.0"
    assert jnp.allclose(outputs, jnp.array([1.0, 3.0, 6.0])), "Outputs should be [1.0, 3.0, 6.0]"
    
    print("Scan test passed!")
    
test_flax_scan()

AttributeError: 'EvalTrace' object has no attribute 'level'

In [3]:
import flax
import flax.linen as nn
from jax import random

class SimpleScan(nn.Module):
  @nn.compact
  def __call__(self, c, xs):
    LSTM = nn.scan(nn.LSTMCell,
                   variable_broadcast="params",
                   split_rngs={"params": False},
                   in_axes=1,
                   out_axes=1)
    return LSTM()(c, xs)

seq_len, batch_size, in_feat, out_feat = 20, 16, 3, 5
key_1, key_2, key_3 = random.split(random.PRNGKey(0), 3)

xs = random.uniform(key_1, (batch_size, seq_len, in_feat))
init_carry = nn.LSTMCell.initialize_carry(key_2, (batch_size,), out_feat)

model = SimpleScan()
variables = model.init(key_3, init_carry, xs)
out_carry, out_val = model.apply(variables, init_carry, xs)

assert out_val.shape == (batch_size, seq_len, out_feat)

TypeError: 'int' object is not subscriptable

In [16]:
input_shape

NameError: name 'input_shape' is not defined